# Práctica 2 · Tautomería y protonación como microespacio químico
### Del SMILES a la distribución de Boltzmann

| Campo | Valor |
|:--|:--|
| **Bloque** | 1 — Generación de estructuras y espacio conformacional |
| **Nivel** | Básico |
| **Tiempo estimado** | 1.5 h (semilla) + 2 h (bosque y análisis) |
| **Pipeline** | SMILES → tautómeros → geometría 3D → GFN2-xTB → ΔG → Boltzmann → microespacio |
| **Prerrequisito** | Práctica 1 completada |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qcmanual/del-orbital-al-espacio-quimico/blob/main/notebooks/p02.ipynb)


---
## Introducción

Cuando un químico escribe el SMILES de una molécula, implícitamente elige *una* forma
tautomérica y *un* estado de protonación. En condiciones reales —en solución, en el
sitio activo de una enzima— la misma molécula puede existir simultáneamente en
**múltiples formas que se interconvierten rápidamente**: se habla de *especiación química*.

Esta especiación tiene consecuencias directas sobre la reactividad, la solubilidad y
la afinidad de unión a proteínas. Si el cálculo se ejecuta sobre el tautómero incorrecto,
todos los resultados subsecuentes son los de una especie que puede ser minoritaria
o inexistente en las condiciones de interés.

### Pipeline de la práctica

```
SMILES semilla
     │
     ▼
 Enumeración de tautómeros     (RDKit TautomerEnumerator)
     │
     ▼
 Geometría 3D para cada forma  (ETKDG v3 + MMFF94)
     │
     ▼
 Optimización + frecuencias    (GFN2-xTB, --opt tight --ohess)
     │
     ▼
 ΔG relativo entre formas      (1 Hartree = 2625.5 kJ/mol)
     │
     ▼
 Distribución de Boltzmann  →  microespacio químico
```

> **🌱 Semilla:** 2-piridona / 2-hidroxipiridina — el par tautómero clásico de la
> química heterocíclica, con valor experimental de referencia (Beak, 1976).
>
> **🌳 Bosque:** 40 moléculas con tautomería documentada (bases nucleicas, fármacos,
> heterociclos, sistemas ceto-enol).


---
## Cuadro conceptual: los tres lenguajes lineales de moléculas

Antes de escribir código es necesario entender cómo se representan las moléculas
como texto. Los tres sistemas que aparecen en esta práctica son:

---

### SMILES — *Simplified Molecular Input Line Entry System*

SMILES codifica la *topología* de una molécula (qué átomos hay y cómo están
conectados) como una cadena de texto. La molécula se recorre como un grafo en
profundidad: las ramas se marcan con paréntesis y los cierres de anillo con números.

| Fragmento SMILES | Significado |
|:--|:--|
| `C`, `N`, `O` | carbono, nitrógeno, oxígeno (sp3 implícito) |
| `c`, `n`, `o` | átomos aromáticos |
| `=`, `#` | enlace doble, triple |
| `()` | ramificación |
| `1`, `2`… | cierre de anillo |
| `[H]`, `[NH+]` | hidrógenos o cargas explícitos |

Los dos SMILES de esta práctica:
```
O=c1ccccn1H   →  2-piridona        (forma ceto/lactama)
Oc1ccccn1     →  2-hidroxipiridina  (forma enol)
```

> ⚠️ `O=c1ccccn1H` y `Oc1ccccn1` son el *mismo compuesto* pero **formas tautómeras
> distintas**. El SMILES canónico (generado por `Chem.MolToSmiles()`) produce siempre
> la misma cadena para una molécula dada, sin importar cómo se escribió el SMILES
> de entrada. Esto es lo que permite comparar tautómeros de forma inequívoca.

---

### SMARTS — *SMILES Arbitrary Target Specification*

SMARTS es un lenguaje de **patrones**: mientras SMILES describe *una* molécula
específica, SMARTS describe *una clase* de subestructuras. Es la base de todas
las búsquedas de subestructura en quimioinformática.

El algoritmo `TautomerEnumerator` de RDKit funciona internamente con reglas SMARTS:
pares (patrón reactivo → patrón producto) que definen cada transformación tautómera.
Por ejemplo, la regla ceto→enol:
```
[CX4:1][C:2]=O  >>  [C:1]=[C:2][OH]     # carbono sp3 alpha a un carbonilo
```

| Símbolo SMARTS | Significado |
|:--|:--|
| `[CX4]` | carbono con exactamente 4 vecinos (sp3) |
| `[#6]` | cualquier átomo de número atómico 6 (carbono) |
| `[!H]` | átomo sin hidrógenos implícitos |
| `[C:1]` | carbono etiquetado como átomo 1 (para capturar su índice) |

> **Limitación clave:** si un tipo de tautomería no está en las reglas de Sitzmann
> que usa RDKit, el enumerador no lo generará. Esto es el núcleo de la Pregunta
> de Discusión 1.

---

### SELFIES — *Self-Referencing Embedded Strings*

SELFIES (Krenn et al., 2020) fue diseñado para modelos generativos de IA:
garantiza que **cualquier cadena bien formada corresponde a una molécula válida**.
En SMILES, una cadena aleatoria casi seguramente es inválida.
En SELFIES, la gramática lo hace imposible por construcción.

En esta práctica usamos SMILES (más legibles y plenamente soportados por RDKit).
SELFIES aparece en el Bloque 5 cuando trabajemos con modelos generativos.

> 📌 **Espacio para figura** *(insertar aquí)*: diagrama de cuatro cuadrantes
> comparando los tres sistemas con el mismo ejemplo: estructura 2D de la 2-piridona,
> su SMILES, el SMARTS que la captura, y su representación SELFIES.


---
## Marco teórico

### 1. Equilibrio tautomérico

Dos moléculas son **tautómeros** si se interconvierten por migración de un protón
con desplazamiento concomitante de un enlace doble. El equilibrio entre T₁ y T₂
está gobernado por:

$$K_T = \frac{[T_2]}{[T_1]} = \exp\!\left(-\frac{\Delta G^\circ}{RT}\right)$$

### 2. Distribución de Boltzmann

Para $n$ tautómeros con energías libres $G_i$:

$$p_i = \frac{e^{-\Delta G_i / RT}}{\sum_j e^{-\Delta G_j / RT}}$$

donde $\Delta G_i = G_i - G_\text{min}$ es la energía relativa al tautómero más estable.
Trabajar con $\Delta G$ (no con $G$ absoluto) evita problemas numéricos con
exponentes grandes y hace la expresión independiente del origen de energías.

| $\Delta G$ (kJ/mol) | $p_\text{minoritario}$ | Significado práctico |
|:--:|:--:|:--|
| 0 | 50 % | formas equimolares |
| 5.7 | ~10 % | detectable en RMN |
| 11.4 | ~1 % | apenas detectable |
| 17.1 | ~0.1 % | relevante para mutaciones |
| > 23 | < 0.01 % | prácticamente indetectable |

A 298 K, $RT = 2.479$ kJ/mol. Una diferencia de apenas $5RT$ reduce la forma
minoritaria al ~1 %.

### 3. GFN2-xTB y la energía libre de Gibbs

Con la bandera `--ohess`, xTB calcula las correcciones termodinámicas en la
geometría optimizada:

$$G(T) = E_\text{elec} + E_\text{ZPE} + H_\text{vib}(T) - TS_\text{vib}(T)
+ H_\text{rot+trans} - TS_\text{rot+trans}$$

Para comparar tautómeros isómeros, las contribuciones traslacionales y rotacionales
se cancelan en gran medida; la diferencia está dominada por
$\Delta E_\text{elec} + \Delta$ZPE. Aun así, siempre se usa $G$ (no $E_\text{elec}$)
porque la diferencia puede ser 1–5 kJ/mol y puede cambiar el orden relativo.

> **Validación de referencia:** Beak (1976) reportó
> $\Delta G^\circ_\text{gas} = -6.3$ kJ/mol a favor de la 2-piridona.
> Error típico de GFN2-xTB para tautómeros de heterociclos: **< 2 kJ/mol**.


---
## ⚙️ Celda 0 — Configuración del entorno

**Ejecuta esta celda primero.** Instala dependencias (solo en Colab),
importa todas las librerías y define las constantes globales que
se usan en el resto del notebook.


In [ ]:
# ── Instalación (solo en Google Colab) ──────────────────────────────────────
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'rdkit', 'py3Dmol', 'plotly', 'kaleido', 'scipy'],
                   check=False)

# ── Imports ───────────────────────────────────────────────────────────────────
import os, re, time, warnings, pathlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

# RDKit — quimioinformática
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw, rdmolfiles, rdMolDescriptors
    from rdkit.Chem.Draw import MolsToGridImage
    from rdkit.Chem.MolStandardize import rdMolStandardize
    print('✓ RDKit', Chem.__version__ if hasattr(Chem, '__version__') else 'OK')
    RDKIT_OK = True
except ImportError:
    print('✗ RDKit — instala: conda install -c conda-forge rdkit')
    RDKIT_OK = False

# py3Dmol — visualización 3D interactiva en el navegador (WebGL)
try:
    import py3Dmol
    print('✓ py3Dmol OK')
    PY3DMOL_OK = True
except ImportError:
    print('⚠ py3Dmol no disponible — las celdas 3D darán instrucciones para Avogadro2')
    PY3DMOL_OK = False

# Plotly — tablero interactivo del bosque
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print('✓ Plotly OK')
    PLOTLY_OK = True
except ImportError:
    print('⚠ Plotly no disponible — instala: pip install plotly')
    PLOTLY_OK = False

# scipy — correlación de Pearson en la figura de paridad
try:
    from scipy import stats as sp_stats
    print('✓ scipy OK')
except ImportError:
    print('⚠ scipy no disponible — instala: pip install scipy')

# xtb — verificar disponibilidad en PATH
XTB_OK = shutil.which('xtb') is not None
print('✓ xtb:', shutil.which('xtb') if XTB_OK else
      '✗ no encontrado — se usarán energías simuladas (fallback pedagógico)')

# ── Constantes globales ───────────────────────────────────────────────────────
# Usadas en todas las celdas de cálculo termodinámico
R     = 8.314e-3   # kJ/(mol·K)
T_K   = 298.15     # K
EH2KJ = 2625.5     # factor de conversión: 1 Hartree → kJ/mol

# ── Variables de la semilla (usadas como referencia global) ───────────────────
SMILES_SEMILLA = 'O=c1ccccn1H'   # 2-piridona, forma ceto (lactama)
NOMBRE         = '2-piridona'
DIRECTORIO     = f'{NOMBRE}_tautomeros'

print(f'\n✓ Configuración lista — R = {R} kJ/(mol·K), T = {T_K} K')


---
## 📝 Preguntas previas

Responde **antes de ejecutar cualquier celda de cálculo**.
Tus respuestas quedan registradas en el notebook; las revisarás al final.


**Pregunta 1 (Conceptual).** La guanina tiene cuatro tautómeros biológicamente
relevantes: ceto-amino (dominante), enol-amino, ceto-imino y enol-imino. Sin calcular
nada, ¿cuál esperas que sea el más estable y por qué? ¿Qué consecuencia tendría
para el apareamiento Watson-Crick si el tautómero minoritario estuviera presente
en el 0.01 % de las moléculas?


In [ ]:
respuesta_1 = (
    'El tautómero más estable de la guanina probablemente es...\n'
    'porque...\n'
    'La consecuencia para Watson-Crick sería...'
)
print('Respuesta 1 guardada.')


**Pregunta 2 (Predictivo).** La 2-piridona puede existir como **forma ceto**
(lactama: C=O + N-H) o como **forma enol** (2-hidroxipiridina: C-OH, N sin H).
¿Cuál esperas que sea más estable en fase gas? ¿Cambiaría ese orden en agua?
¿Qué argumento químico usarías?


In [ ]:
respuesta_2 = (
    'En fase gas espero que sea más estable la forma...\n'
    'porque...\n'
    'En agua cambiaría / no cambiaría porque...'
)
print('Respuesta 2 guardada.')


**Pregunta 3 (Crítico).** El pipeline calcula energías en vacío (fase gas) y aplica
Boltzmann. ¿En qué circunstancias daría predicciones incorrectas sobre qué tautómero
predomina en solución acuosa? ¿Qué habría que añadir al protocolo para corregirlo?


In [ ]:
respuesta_3 = (
    'El protocolo en vacío fallaría cuando...\n'
    'Para corregirlo habría que añadir...'
)
print('Respuesta 3 guardada.')


---
## 🌱 Semilla · Paso 1 — Enumeración de tautómeros con RDKit

### ¿Qué hace `TautomerEnumerator`?

`TautomerEnumerator` aplica las reglas de transformación de Sitzmann, codificadas
como pares de SMARTS (patrón reactivo → patrón producto). Por ejemplo:
```
[CX4:1][C:2]=O  >>  [C:1]=[C:2][OH]     # regla ceto → enol
```
El algoritmo aplica todas las reglas válidas de forma combinatoria y elimina
duplicados usando el SMILES canónico como clave de identidad.

`SetMaxTautomers(100)` pone un límite de seguridad: para moléculas con muchos
grupos tautomerizables el número de formas puede crecer exponencialmente.

> 📌 **Espacio para figura** *(insertar aquí)*: las dos estructuras Lewis de la
> 2-piridona y la 2-hidroxipiridina con el protón migrante resaltado y las
> longitudes de enlace C=O / C–OH anotadas.

### Instrucciones
1. Ejecuta la celda. Observa la rejilla: identifica la **forma ceto** (C=O + N-H)
   y la **forma enol** (C-OH, N sin H).
2. ¿Hay alguna forma que parezca químicamente improbable? Anota cuál y por qué.


In [ ]:
# ── Paso 1: SMILES → tautómeros → rejilla 2D ─────────────────────────────────
os.makedirs(DIRECTORIO, exist_ok=True)

# MolFromSmiles convierte la cadena de texto en un grafo molecular (nodos = átomos,
# aristas = enlaces). Retorna None si el SMILES no es sintácticamente válido.
mol = Chem.MolFromSmiles(SMILES_SEMILLA)
if mol is None:
    raise ValueError(f'SMILES no válido: {SMILES_SEMILLA}')

print(f'Molécula         : {NOMBRE}')
print(f'SMILES canónico  : {Chem.MolToSmiles(mol)}')
print(f'Fórmula          : {rdMolDescriptors.CalcMolFormula(mol)}')
print()

# Configurar el enumerador con las reglas de Sitzmann
enumerador = rdMolStandardize.TautomerEnumerator()
enumerador.SetMaxTautomers(100)
enumerador.SetRemoveSp3Stereo(True)  # ignorar estereoquímica sp3 al comparar

# Enumerate() devuelve una lista de objetos Mol — uno por tautómero
tautomeros = list(enumerador.Enumerate(mol))
n_tau = len(tautomeros)
print(f'Tautómeros encontrados: {n_tau}')
print()

# MolToSmiles genera el SMILES canónico de cada tautómero.
# Aunque dos strings de entrada sean distintos, si representan la misma
# molécula producen el mismo SMILES canónico → así se eliminan duplicados.
for i, tau in enumerate(tautomeros):
    print(f'  T{i+1:02d}: {Chem.MolToSmiles(tau)}')

print()

# MolsToGridImage dibuja todos los tautómeros en una cuadrícula PNG.
# La numeración T01, T02... se usa de aquí en adelante en todos los archivos.
img = MolsToGridImage(
    tautomeros,
    molsPerRow=4,
    subImgSize=(320, 260),
    legends=[f'T{i+1:02d}: {Chem.MolToSmiles(t)}' for i, t in enumerate(tautomeros)]
)
img.save(f'{NOMBRE}_tautomeros_2D.png')
display(img)


---
### 🔍 Demo: SMARTS como detector de patrones tautómeros

Antes de continuar, confirmamos con búsquedas SMARTS que los tautómeros
generados son genuinamente distintos. Esto ilustra también por qué
`TautomerEnumerator` *sabe* que la molécula puede tautomerizarse:
detecta internamente estos mismos patrones.


In [ ]:
# ── Demo SMARTS: detectar patrones que distinguen cada tautómero ──────────────
# HasSubstructMatch() devuelve True si la molécula contiene el patrón dado.
# Los patrones son SMARTS, no SMILES — el lenguaje de búsqueda de RDKit.

patrones = {
    'Carbonilo en anillo [NX3][CX3]=O'  : '[NX3][CX3]=O',
    'Enol genérico [CX3]=[CX3][OH]'     : '[CX3]=[CX3][OH]',
    'N-H en anillo aromático [nH]'       : '[nH]',
    'N piridínico sin H [n;!H1]'         : '[n;!H1]',
}

encabezado = '  '.join([f'T{i+1}' for i in range(n_tau)])
print(f'{'Patrón SMARTS':<42}  {encabezado}')
print('─' * (44 + 4 * n_tau))
for descripcion, smarts_str in patrones.items():
    patron = Chem.MolFromSmarts(smarts_str)
    if patron is None:
        continue
    matches = '   '.join(['✓' if t.HasSubstructMatch(patron) else '✗'
                          for t in tautomeros])
    print(f'  {descripcion:<40}  {matches}')

print()
print('Los distintos patrones en T1 vs T2 confirman que son formas tautómeras genuinas.')
print('TautomerEnumerator detecta estos mismos patrones internamente para decidir')
print('qué reglas de transformación son aplicables.')


---
## 🌱 Semilla · Paso 2 — Geometría 3D para cada tautómero (ETKDG v3 + MMFF94)

### ¿Qué hace este paso?

Hasta aquí cada tautómero es solo un grafo 2D — no tiene coordenadas cartesianas.
El proceso de generación de geometría 3D tiene dos etapas:

1. **ETKDG v3 (Experimental Torsion-angle Knowledge Distance Geometry):** genera
   coordenadas iniciales resolviendo restricciones de distancia extraídas de
   estructuras de cristal. El `randomSeed=42` garantiza reproducibilidad.
2. **MMFF94 (Merck Molecular Force Field):** minimiza las tensiones geométricas
   antes de pasar a xTB. Si MMFF94 no tiene parámetros para un átomo, se usa
   UFF como alternativa.

> **Sutileza importante:** cada tautómero tiene conectividad diferente — el H
> migratorio ocupa un lugar distinto en el grafo. Por tanto, cada forma necesita
> su propia geometría 3D independiente. No basta con mover el H de una geometría
> optimizada a otra posición.

### Instrucciones
1. Ejecuta la celda. Se genera un archivo `_FF.xyz` por tautómero.
2. Observa si algún tautómero falla en la incrustación. ¿Por qué crees que falla?
3. Compara las energías MMFF94: ¿ya se ordenan como esperas?
   MMFF94 no captura efectos electrónicos finos — por eso necesitamos xTB.


In [ ]:
# ── Paso 2: geometría 3D para cada tautómero ─────────────────────────────────
registros_geom = []

for i, tau in enumerate(tautomeros):
    etiqueta = f'{NOMBRE}_T{i+1:02d}'

    # AddHs: hidrógenos explícitos necesarios para ETKDG y para xTB (--ohess)
    tau_h = Chem.AddHs(tau)

    # ETKDGv3: ok=0 éxito, ok=-1 fallo (molécula muy tensionada)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    ok = AllChem.EmbedMolecule(tau_h, params)

    if ok == -1:
        print(f'  T{i+1:02d}: ✗ fallo en incrustación ETKDG')
        registros_geom.append({'etiqueta': etiqueta,
                                'smiles': Chem.MolToSmiles(tau),
                                'embed_ok': False, 'e_ff_kcalmol': None})
        continue

    props = AllChem.MMFFGetMoleculeProperties(tau_h)
    ff    = AllChem.MMFFGetMoleculeForceField(tau_h, props)
    if ff is None:
        print(f'  T{i+1:02d}: ⚠ MMFF94 sin parámetros, usando UFF')
        ff = AllChem.UFFGetMoleculeForceField(tau_h)
    ff.Minimize(maxIts=2000)
    e_ff = ff.CalcEnergy()  # energía final en kcal/mol

    xyz_path = os.path.join(DIRECTORIO, f'{etiqueta}_FF.xyz')
    rdmolfiles.MolToXYZFile(tau_h, xyz_path)

    registros_geom.append({'etiqueta': etiqueta,
                            'smiles': Chem.MolToSmiles(tau),
                            'embed_ok': True, 'e_ff_kcalmol': round(e_ff, 3)})
    print(f'  T{i+1:02d}: ✓  E_MMFF94 = {e_ff:8.2f} kcal/mol')

df_geom = pd.DataFrame(registros_geom)
n_ok = df_geom['embed_ok'].sum()
print(f'\n{n_ok}/{n_tau} tautómeros con geometría lista para xTB')


---
## 🌱 Semilla · Paso 3 — Optimización + frecuencias con GFN2-xTB

### Flags del comando

```bash
xtb T01_FF.xyz --opt tight --ohess --chrg 0 --uhf 0 --gfn 2 --T 298.15
```

| Flag | Significado |
|:--|:--|
| `--opt tight` | Convergencia estricta: gradiente RMS < 2×10⁻⁴ Eh/Bohr |
| `--ohess` | Optimización + hessiano en la geometría final (un solo paso) |
| `--chrg 0` | Molécula neutra |
| `--uhf 0` | Capa cerrada (0 electrones desapareados) |
| `--gfn 2` | Hamiltoniano GFN2-xTB (el más preciso de la familia GFN) |
| `--T 298.15` | Temperatura para las correcciones termodinámicas |

> **¿Por qué `--ohess` y no `--hess`?** `--hess` calcula frecuencias en la
> geometría de *entrada*. `--ohess` las calcula en la geometría *optimizada*.
> Son equivalentes solo si la entrada ya es un mínimo; en la práctica, siempre
> se usa `--ohess` para garantizar consistencia.

> **⚠️ Frecuencia imaginaria:** Si ves un valor `ν < 0` en el log, la geometría
> es un **punto de silla**, no un mínimo. Ese tautómero no puede usarse en la
> distribución de Boltzmann porque su $G$ no está bien definido.

### Instrucciones
1. Ejecuta la celda (1–5 min). Revisa qué tautómeros convergieron.
2. En la celda siguiente, inspecciona el log de T01: ubica las líneas
   `TOTAL ENERGY` y `TOTAL FREE ENERGY`. ¿Cuánto difieren en kJ/mol?
3. Verifica que ningún tautómero tenga frecuencias imaginarias.


In [ ]:
# ── Paso 3a: optimización GFN2-xTB ──────────────────────────────────────────
# xTB se invoca como subproceso externo; Python lanza el proceso, captura
# su salida completa (stdout + stderr) y la guarda para análisis.

if XTB_OK:
    print('xtb disponible — ejecutando optimizaciones reales...\n')
    for xyz_file in sorted(os.listdir(DIRECTORIO)):
        if not xyz_file.endswith('_FF.xyz'):
            continue
        base    = xyz_file.replace('_FF.xyz', '')
        log_out = os.path.join(DIRECTORIO, f'{base}_xtb.out')
        result  = subprocess.run(
            ['xtb', xyz_file,
             '--opt', 'tight', '--ohess',
             '--chrg', '0', '--uhf', '0',
             '--gfn', '2', '--T', '298.15'],
            capture_output=True, text=True, cwd=DIRECTORIO
        )
        with open(log_out, 'w') as f:
            f.write(result.stdout)
        # La geometría optimizada se escribe en xtbopt.xyz → renombrar
        opt_src = os.path.join(DIRECTORIO, 'xtbopt.xyz')
        if os.path.exists(opt_src):
            os.rename(opt_src, os.path.join(DIRECTORIO, f'{base}_opt.xyz'))
        convergio = 'GEOMETRY OPTIMIZATION CONVERGED' in result.stdout
        print(f'  {base}: {"✓ convergió" if convergio else "⚠ no convergió"}')

else:
    # ── Fallback pedagógico: logs simulados con valores de referencia ─────────
    # Las energías son consistentes con ΔG ≈ −6.3 kJ/mol (Beak, 1976).
    # El flujo de análisis de las celdas siguientes es idéntico al caso real.
    print('xtb no disponible — generando logs simulados (valores de Beak, 1976):\n')
    # −6.3 kJ/mol / 2625.5 kJ/mol·Eh⁻¹ = 0.002400 Eh
    sims = [
        (f'{NOMBRE}_T01', -19.887300),   # 2-piridona (forma dominante)
        (f'{NOMBRE}_T02', -19.884900),   # 2-hidroxipiridina (+6.3 kJ/mol)
    ]
    for base, g_val in sims:
        log_txt = (
            f'GFN2-xTB [SIMULADO — fallback pedagógico]\n'
            f'GEOMETRY OPTIMIZATION CONVERGED\n'
            f'TOTAL ENERGY      {g_val + 0.002400:.8f} Eh\n'
            f'TOTAL FREE ENERGY {g_val:.8f} Eh\n'
            f'zero point energy        0.0623 Eh\n'
        )
        with open(os.path.join(DIRECTORIO, f'{base}_xtb.out'), 'w') as f:
            f.write(log_txt)
        src = os.path.join(DIRECTORIO, f'{base}_FF.xyz')
        dst = os.path.join(DIRECTORIO, f'{base}_opt.xyz')
        if os.path.exists(src):
            shutil.copy(src, dst)
        print(f'  {base}: log simulado guardado.')

print('\n✓ Paso 3a completado.')


In [ ]:
# ── Paso 3b: anatomía de un log xTB ──────────────────────────────────────────
# Leer logs de programas de QC es una habilidad fundamental.
# Gaussian, ORCA y xTB producen archivos con estructura similar.
# Aquí extraemos las líneas más importantes del log de T01.

log_t01 = os.path.join(DIRECTORIO, f'{NOMBRE}_T01_xtb.out')
if pathlib.Path(log_t01).exists():
    with open(log_t01) as f:
        log = f.read()

    print('═' * 55)
    print('  EXTRACTO DEL LOG xTB — T01')
    print('═' * 55)

    # Líneas clave del log
    for kw in ['GEOMETRY OPTIMIZATION CONVERGED',
               'TOTAL ENERGY', 'TOTAL FREE ENERGY', 'zero point energy']:
        for line in log.splitlines():
            if kw in line:
                print(f'  {line.strip()}')
                break

    # E_elec vs G(298 K): la diferencia son las correcciones termodinámicas
    m_e = re.search(r'TOTAL ENERGY\s+([-\d.]+)', log)
    m_g = re.search(r'TOTAL FREE ENERGY\s+([-\d.]+)', log)
    if m_e and m_g:
        E = float(m_e.group(1)); G = float(m_g.group(1))
        print(f'\n  G − E (correcciones ZPE+vib+rot+trans) = {(G-E)*EH2KJ:+.2f} kJ/mol')
        print('  → Para comparar tautómeros isómeros, siempre usar TOTAL FREE ENERGY.')

    # Frecuencias imaginarias: si n > 0, la geometría no es un mínimo
    n_imag = len(re.findall(r'\s+-\d+\.\d+\s+cm', log))
    print(f'\n  Frecuencias imaginarias: {n_imag}'
          f' → {"mínimo real ✓" if n_imag == 0 else "punto de silla ⚠"}')
else:
    print(f'Log no encontrado: {log_t01}. Ejecuta el Paso 3a primero.')


In [ ]:
# ── Paso 3c: control de calidad — verificar todos los tautómeros ──────────────
# Criterio: convergencia = True y frecuencias imaginarias = 0.
# Los que no pasen se excluyen de la distribución de Boltzmann.

registros_xtb = []
for arch in sorted(os.listdir(DIRECTORIO)):
    if not arch.endswith('_xtb.out'):
        continue
    with open(os.path.join(DIRECTORIO, arch)) as f:
        log = f.read()
    etiqueta   = arch.replace('_xtb.out', '')
    convergio  = 'GEOMETRY OPTIMIZATION CONVERGED' in log
    m_e        = re.search(r'TOTAL ENERGY\s+([-\d.]+)', log)
    m_g        = re.search(r'TOTAL FREE ENERGY\s+([-\d.]+)', log)
    n_imag     = len(re.findall(r'\s+-\d+\.\d+\s+cm', log))
    registros_xtb.append({
        'etiqueta'   : etiqueta,
        'convergio'  : convergio,
        'E_total_Eh' : float(m_e.group(1)) if m_e else None,
        'G_298_Eh'   : float(m_g.group(1)) if m_g else None,
        'n_imag'     : n_imag,
    })

df_xtb = pd.DataFrame(registros_xtb)

print('=== Control de calidad ===')
for _, row in df_xtb.iterrows():
    estado = '✓' if (row['convergio'] and row['n_imag'] == 0) else '⚠'
    print(f'  {estado} {row["etiqueta"]}: '
          f'convergió={row["convergio"]}, freq_imag={row["n_imag"]}')

df_validos = df_xtb[df_xtb['convergio'] & (df_xtb['n_imag'] == 0)].dropna(subset=['G_298_Eh'])
print(f'\nTautómeros válidos para Boltzmann: {len(df_validos)}/{len(df_xtb)}')


---
## 🌱 Semilla · Paso 4 — Distribución de Boltzmann

### ¿Qué hace este paso?

Convierte las energías libres $G_i$ de cada tautómero en fracciones molares
de equilibrio. Se calcula $\Delta G_i = G_i - G_\text{min}$ (relativo al más
estable) para mayor estabilidad numérica, y luego:

$$p_i = \frac{e^{-\Delta G_i / RT}}{\sum_j e^{-\Delta G_j / RT}}$$

### Validación vs experimento

| Valor | Referencia (Beak, 1976) | Este cálculo |
|:--|:--|:--|
| $\Delta G^\circ_\text{gas}$ (kJ/mol) | −6.3 | *(ver abajo)* |
| $p$(2-piridona) en gas (%) | ~93 | *(ver abajo)* |

Criterio de aceptación: error < 3 kJ/mol respecto al experimental.

**Formula tu predicción antes de ejecutar:** ¿qué porcentaje esperas
para la forma ceto? ¿Coincide con tu respuesta a la Pregunta 2?


In [ ]:
# ── Paso 4: distribución de Boltzmann ───────────────────────────────────────

df_boltz = df_validos.copy()

# ΔG relativo: el tautómero más estable tiene ΔG = 0 por definición
G_min = df_boltz['G_298_Eh'].min()
df_boltz['dG_kJ'] = (df_boltz['G_298_Eh'] - G_min) * EH2KJ

# Pesos de Boltzmann: el tautómero con ΔG=0 tiene peso=1 (el máximo)
df_boltz['peso'] = np.exp(-df_boltz['dG_kJ'] / (R * T_K))
df_boltz['pct']  = df_boltz['peso'] / df_boltz['peso'].sum() * 100
df_boltz = df_boltz.sort_values('dG_kJ').reset_index(drop=True)

print('=== Distribución de Boltzmann a 298 K ===')
print(f'  {"Tautómero":<28}  {"G (Eh)":>13}  {"ΔG (kJ/mol)":>12}  {"p (%)">:7}')
print('  ' + '─' * 65)
for _, row in df_boltz.iterrows():
    marca = '  ← dominante' if row['dG_kJ'] < 0.001 else ''
    print(f'  {row["etiqueta"]:<28}  {row["G_298_Eh"]:>13.8f}'
          f'  {row["dG_kJ"]:>12.3f}  {row["pct"]:>7.2f}%{marca}')

# Validación vs Beak (1976)
if len(df_boltz) > 1:
    dG_calc = df_boltz['dG_kJ'].iloc[1]
    error   = abs(dG_calc - 6.3)
    marca   = '✓' if error < 3 else '⚠'
    print(f'\n=== Validación vs Beak (1976) ===')
    print(f'  ΔG calculado  : {dG_calc:.2f} kJ/mol')
    print(f'  ΔG exp. (gas) :  6.3 kJ/mol')
    print(f'  Error         : {error:.2f} kJ/mol  {marca}')
    print(f'  Criterio (< 3 kJ/mol): {"cumplido" if error < 3 else "no cumplido — revisar"}')

df_boltz.to_csv(f'{NOMBRE}_tautomeros_resultados.csv', index=False)
print(f'\n✓ Guardado: {NOMBRE}_tautomeros_resultados.csv')


---
## 🌱 Semilla · Paso 5 — Figura del equilibrio tautómero

Esta figura comunica el resultado central de la semilla:
qué forma domina y por cuánto la supera energéticamente.

- **Panel izquierdo:** ΔG relativo con el porcentaje de cada forma
- **Panel derecho:** distribución circular de poblaciones de Boltzmann


In [ ]:
# ── Paso 5: figura ΔG + distribución de poblaciones ─────────────────────────
if len(df_boltz) == 0:
    print('Sin datos. Ejecuta los Pasos 3 y 4 primero.')
else:
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    etiq_cortas = df_boltz['etiqueta'].str.replace(f'{NOMBRE}_', '', regex=False)

    # Panel izquierdo: barras de ΔG
    colores = ['#0F2747' if dg < 3 else '#2A6F97' if dg < 10 else '#AABBD0'
               for dg in df_boltz['dG_kJ']]
    bars = ax.bar(etiq_cortas, df_boltz['dG_kJ'],
                  color=colores, edgecolor='white', linewidth=0.5)
    for bar, pct in zip(bars, df_boltz['pct']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Tautómero', fontsize=11)
    ax.set_ylabel('ΔG (kJ/mol)', fontsize=11)
    ax.set_title('Energía libre relativa', fontsize=11)
    ax.axhline(0, color='gray', linewidth=0.5)

    # Panel derecho: pie de poblaciones
    colores_pie = ['#0F2747', '#2A6F97', '#5A9BBF', '#AABBD0', '#D4E5F5']
    _, _, autotexts = ax2.pie(
        df_boltz['pct'], labels=etiq_cortas,
        autopct='%1.1f%%', colors=colores_pie[:len(df_boltz)],
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.2}
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax2.set_title('Distribución de Boltzmann (298 K)', fontsize=11)

    plt.tight_layout()
    plt.savefig('p02_boltzmann_semilla.pdf', bbox_inches='tight')
    plt.show()
    print('✓ Guardada: p02_boltzmann_semilla.pdf')


---
## 🌡️ Exploración — ¿Cómo cambia el equilibrio con la temperatura?

La distribución de Boltzmann depende de $T$: a temperaturas altas las
poblaciones se igualan; a temperaturas bajas el tautómero más estable
domina casi completamente.

Esta celda calcula y grafica las poblaciones entre 200 y 800 K usando
los $\Delta G$ obtenidos en el Paso 4, sin necesidad de recalcular xTB.

> **Aproximación:** se mantienen los mismos $\Delta G$ calculados a 298 K.
> Válida si $\Delta C_p \approx 0$, razonable para tautómeros de heterociclos
> en el rango 200–800 K.


In [ ]:
# ── Exploración: p_i(T) entre 200 y 800 K ───────────────────────────────────
if len(df_boltz) == 0:
    print('Sin datos. Ejecuta los Pasos 3 y 4 primero.')
else:
    temperaturas = np.linspace(200, 800, 200)
    dG_array     = df_boltz['dG_kJ'].values

    # Para cada temperatura recalculamos pesos y normalizamos
    pop_vs_T = np.array([
        np.exp(-dG_array / (R * T_i)) / np.exp(-dG_array / (R * T_i)).sum() * 100
        for T_i in temperaturas
    ])

    fig, ax = plt.subplots(figsize=(9, 5))
    colores_t = ['#0F2747', '#C03B26', '#5A9BBF', '#7BB37D', '#E8A838']
    for j, (_, row) in enumerate(df_boltz.iterrows()):
        etiq = row['etiqueta'].replace(f'{NOMBRE}_', '')
        ax.plot(temperaturas, pop_vs_T[:, j],
                color=colores_t[j % len(colores_t)],
                linewidth=2, label=f'{etiq} ({row["pct"]:.1f}% @ 298 K)')

    ax.axvline(298.15, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(302, 5, '298 K', color='gray', fontsize=9)
    ax.set_xlabel('Temperatura (K)', fontsize=12)
    ax.set_ylabel('Población (%)', fontsize=12)
    ax.set_title('Distribución de Boltzmann vs temperatura\n'
                 '(ΔG congelado a 298 K — aproximación)', fontsize=12)
    ax.legend(fontsize=9, loc='center right')
    ax.set_xlim(200, 800); ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.savefig('p02_pop_vs_T.pdf', bbox_inches='tight')
    plt.show()

    if len(df_boltz) >= 2:
        # Temperatura a la que p1 = p2 = 50% (equipartición entre los dos más estables)
        T_eq = df_boltz['dG_kJ'].iloc[1] / (R * np.log(2))
        print(f'Temperatura de equipartición (p₁ = p₂ = 50 %): ~{T_eq:.0f} K')

    print('\nPreguntas de reflexión:')
    print('  · ¿A qué temperatura el tautómero minoritario alcanzaría el 10 %?')
    print('  · ¿Es esa temperatura químicamente accesible sin descomponer la molécula?')
    print('  · ¿Qué implica la gráfica para experimentos a temperatura alta?')
    print('✓ Guardada: p02_pop_vs_T.pdf')


---
## 🌱 Semilla · Paso 6 — Visualización 3D comparativa

Antes de extraer parámetros geométricos hay que **ver** la geometría.
Una inspección visual detecta en segundos errores graves —molécula
'explotada', H en posición absurda, anillo no plano— que ningún
número en el output indicaría directamente.

Aquí comparamos T1 (2-piridona) y T2 (2-hidroxipiridina) para confirmar
visualmente que son estructuralmente distintas. Observa especialmente:
- La posición del **hidrógeno móvil** (en N o en O)
- La diferencia visible en la longitud C=O vs C–OH

### Instrucciones
1. Ejecuta la celda. Rota las moléculas (clic + arrastrar).
2. Identifica en cuál forma el H está sobre el N y en cuál sobre el O.
3. ¿El anillo parece plano? ¿Qué implica eso sobre la aromaticidad?
   (El Paso 7 lo cuantifica.)


In [ ]:
# ── Paso 6: visualización 3D con py3Dmol ────────────────────────────────────
# py3Dmol renderiza moléculas 3D directamente en el notebook usando WebGL.
# Verde = T1 (dominante), naranja = T2 (segunda forma más estable).

def ver_xyz(archivo, titulo='', ancho=480, alto=360):
    """Muestra un archivo .xyz con py3Dmol. Si no está disponible, sugiere Avogadro2."""
    if not PY3DMOL_OK:
        print(f'py3Dmol no disponible. Abre {archivo} en Avogadro2.')
        return
    p = pathlib.Path(archivo)
    if not p.exists():
        print(f'Archivo no encontrado: {archivo}')
        return
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModel(p.read_text(), 'xyz')
    v.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.22}})
    v.setBackgroundColor('white')
    if titulo:
        v.addLabel(titulo, {'position': {'x': 0, 'y': 2, 'z': 0},
                            'fontColor': 'black', 'fontSize': 13,
                            'backgroundColor': 'rgba(255,255,255,0.7)'})
    v.zoomTo(); v.show()

# Buscar geometrías optimizadas disponibles
archivos_opt = sorted([f for f in os.listdir(DIRECTORIO) if f.endswith('_opt.xyz')])

if PY3DMOL_OK and len(archivos_opt) >= 2:
    viewer = py3Dmol.view(width=720, height=360, linked=True, viewergrid=(1, 2))
    estilos = [
        {'stick': {'colorscheme': 'greenCarbon',  'radius': 0.15}, 'sphere': {'scale': 0.25}},
        {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.15}, 'sphere': {'scale': 0.25}},
    ]
    for idx, (arch, estilo) in enumerate(zip(archivos_opt[:2], estilos)):
        contenido = pathlib.Path(os.path.join(DIRECTORIO, arch)).read_text()
        etiq      = arch.replace('_opt.xyz', '').replace(f'{NOMBRE}_', '')
        viewer.addModel(contenido, 'xyz', viewer=(0, idx))
        viewer.setStyle(estilo, viewer=(0, idx))
        viewer.addLabel(etiq, {'position': {'x': 0, 'y': 2.5, 'z': 0},
                                'fontSize': 14, 'fontColor': 'black'}, viewer=(0, idx))
        viewer.zoomTo(viewer=(0, idx))
    viewer.show()
    print('Verde: T1 (2-piridona, dominante)  |  Naranja: T2 (2-hidroxipiridina)')
    print('Arrastra para rotar. Rueda del ratón para zoom.')
else:
    print('py3Dmol no disponible o faltan geometrías optimizadas.')
    print(f'Abre los archivos *_opt.xyz de la carpeta {DIRECTORIO} en Avogadro2.')


---
## 🌱 Semilla · Paso 7 — Validación geométrica

Más allá de la energía, confirmamos la identidad de cada tautómero
midiendo las distancias de enlace clave:

| Enlace | 2-piridona (ceto) | 2-hidroxipiridina (enol) |
|:--|:--|:--|
| C=O | ~1.22 Å | — |
| C–O | — | ~1.34 Å |
| O–H | — | ~0.97 Å |
| N–H | ~1.01 Å | — |

> 📐 **Para el docente:** insertar aquí un diagrama con las distancias
> de enlace anotadas sobre la estructura 2D de cada tautómero,
> con valores calc vs exp lado a lado.

### ¿Por qué necesitamos SMARTS aquí?

El archivo `.xyz` solo contiene posiciones numeradas: no sabe cuál es
'el carbono del carbonilo'. Para calcular $d$(C=O) necesitamos los
índices exactos de esos átomos.

Usamos el patrón `[C:1](=[O:2])` para pedirle a RDKit que encuentre
todos los C=O y devuelva sus índices. Esas tuplas `(idx_C, idx_O)` se
usan directamente sobre el array de coordenadas del `.xyz`. Este puente
SMARTS ↔ índices XYZ se repite en todas las prácticas de validación.

### Instrucciones
1. Ejecuta las dos celdas.
2. ¿Las distancias confirman que T1 es la forma ceto? ¿Coincide con la Pregunta 2?
3. ¿El RMSD de planaridad es consistente con aromaticidad?


In [ ]:
# ── Funciones de geometría ────────────────────────────────────────────────────
def leer_xyz(archivo):
    """Lee un archivo .xyz → (lista de símbolos, array Nx3 de coordenadas en Å)."""
    with open(archivo) as f:
        lineas = f.readlines()
    n = int(lineas[0])  # primera línea = número de átomos
    simbolos, coords = [], []
    for l in lineas[2:2+n]:  # a partir de la tercera línea: símbolo x y z
        p = l.split()
        simbolos.append(p[0])
        coords.append([float(x) for x in p[1:4]])
    return simbolos, np.array(coords)

def dist_enlace(coords, i, j):
    """Distancia euclidiana entre los átomos i y j (índices base-0), en Å."""
    return float(np.linalg.norm(coords[i] - coords[j]))

def rmsd_planaridad(coords, idx_anillo):
    """RMSD de planaridad: desviación media de los átomos del anillo respecto
    al plano óptimo por mínimos cuadrados. < 0.05 Å indica anillo plano."""
    pts   = coords[np.array(idx_anillo)]
    pts_c = pts - pts.mean(axis=0)
    _, _, Vt = np.linalg.svd(pts_c)   # Vt[-1] = normal al plano óptimo
    return float(np.sqrt(np.mean((pts_c @ Vt[-1])**2)))

print('Funciones cargadas: leer_xyz | dist_enlace | rmsd_planaridad')


In [ ]:
# ── Validación geométrica con SMARTS ─────────────────────────────────────────
if len(df_boltz) == 0:
    print('Sin resultados de Boltzmann. Ejecuta los Pasos 3 y 4 primero.')
else:
    t1_label = df_boltz.iloc[0]['etiqueta']
    t1_smi   = df_boltz.iloc[0]['etiqueta']  # recuperar SMILES del df_xtb
    t1_smi   = df_xtb.set_index('etiqueta').loc[t1_label, 'etiqueta'] \
               if False else df_validos.iloc[0].get('smiles', SMILES_SEMILLA)
    t1_xyz   = os.path.join(DIRECTORIO, f'{t1_label}_opt.xyz')

    if not pathlib.Path(t1_xyz).exists():
        print(f'No se encontró {t1_xyz}. Verifica el Paso 3a.')
    else:
        _, coords = leer_xyz(t1_xyz)

        # Reconstruir el grafo desde SMILES para usar búsquedas SMARTS
        # (el .xyz no tiene información de conectividad, solo posiciones)
        mol_ref   = Chem.AddHs(Chem.MolFromSmiles(SMILES_SEMILLA))
        AllChem.EmbedMolecule(mol_ref, AllChem.ETKDGv3())

        # Pares (SMARTS, descripción, referencia en Å)
        patrones_val = [
            ('[C:1]=[O:2]',  'C=O (carbonilo)',     1.220),
            ('[c:1]-[OH:2]', 'C–OH (fenólico)',      1.340),
            ('[N:1]-[H:2]',  'N–H (lactama)',        1.010),
            ('[O:1]-[H:2]',  'O–H (fenólico)',       0.970),
        ]

        print(f'Tautómero: {t1_label}')
        print(f'  {"Enlace":<22}  {"Calc (Å)":>10}  {"Ref (Å)":>9}  {"Error %":>8}')
        print('  ' + '─' * 55)
        for smarts, desc, ref in patrones_val:
            pat     = Chem.MolFromSmarts(smarts)
            matches = mol_ref.GetSubstructMatches(pat)
            if matches:
                d   = dist_enlace(coords, matches[0][0], matches[0][1])
                err = abs(d - ref) / ref * 100
                ok  = '✓' if err < 3 else '⚠'
                print(f'  {desc:<22}  {d:>10.4f}  {ref:>9.3f}  {err:>7.2f}%  {ok}')
            else:
                print(f'  {desc:<22}  {"no encontrado":>10}  {ref:>9.3f}  ---')

        # RMSD de planaridad del anillo de 6 miembros
        pat_6  = Chem.MolFromSmarts('[n,c,N,C]1[n,c,N,C][n,c,N,C][n,c,N,C][n,c,N,C][n,c,N,C]1')
        match6 = mol_ref.GetSubstructMatches(pat_6)
        if match6:
            rmsd = rmsd_planaridad(coords, list(match6[0]))
            ok_r = '✓' if rmsd < 0.05 else '⚠'
            print(f'\n  RMSD de planaridad del anillo: {rmsd:.4f} Å  {ok_r}')
            print('  < 0.05 Å → anillo plano → consistente con deslocalización π')


---
## 🌳 Parte 2 · El bosque — 40 moléculas con tautomería documentada

El bosque contiene 40 moléculas elegidas para representar la diversidad
de la tautomería en contextos reales:

| Familia | N | Ejemplos |
|:--|:--:|:--|
| Bases nucleicas | 8 | adenina, guanina, citosina, uracilo, hipoxantina |
| Fármacos con tautomería conocida | 12 | omeprazol, pirimetamina, metotrexato |
| Heterociclos simples | 10 | 4-piridona, imidazol, pirazol, triazoles |
| Ceto-enol no obvio | 10 | acetilacetona, malonaldehído, β-cetoésteres |

El bosque ya está precalculado en `p02_bosque_resultados.csv`.
Tu tarea es integrar la semilla (2-piridona) y analizar el conjunto completo.

> 📌 **Espacio para figura** *(insertar aquí)*: panel 4×2 con un representante
> de cada familia, su estructura 2D y su población $p_1$ anotada.

### Estructura del dataset

| Columna | Tipo | Descripción |
|:--|:--|:--|
| `id` | str | nombre del compuesto |
| `familia` | str | categoría estructural |
| `n_tautomeros` | int | número enumerado por RDKit |
| `dG_T2_kJ` | float | ΔG del segundo tautómero más estable (kJ/mol) |
| `pop_T1_pct` | float | población del tautómero dominante (%) |
| `dG_ref_kJ` | float | valor experimental, si existe |


In [ ]:
# ── Cargar el bosque precalculado ────────────────────────────────────────────
BOSQUE_CSV = 'p02_bosque_resultados.csv'

if os.path.exists(BOSQUE_CSV):
    df_bosque = pd.read_csv(BOSQUE_CSV)
    print(f'Bosque cargado: {len(df_bosque)} moléculas')
    print(df_bosque['familia'].value_counts().to_string())
else:
    print(f'No se encontró {BOSQUE_CSV}')
    print('Opciones:')
    print('  a) Descarga el archivo del repositorio del curso')
    print('  b) Genera el bosque ejecutando: python scripts/p02_expansion.py')
    print('  c) Se crea un bosque sintético de demostración (ver abajo)\n')

    # Bosque sintético: mismo flujo de análisis, energías estadísticamente realistas
    np.random.seed(42)
    n = 40
    familias_demo = (['base_nucleica']*8 + ['farmaco']*12 +
                     ['heterociclo']*10  + ['ceto_enol']*10)
    dG_T2 = np.concatenate([np.random.uniform(5, 25, 8),
                             np.random.uniform(2, 15, 12),
                             np.random.uniform(10, 35, 10),
                             np.random.uniform(1, 12, 10)])
    pesos = np.exp(-dG_T2 / (R * T_K))
    pop1  = 100 / (1 + pesos)
    df_bosque = pd.DataFrame({
        'id'          : [f'mol_{i+1:02d}' for i in range(n)],
        'familia'     : familias_demo,
        'n_tautomeros': np.random.randint(2, 8, n),
        'dG_T2_kJ'    : dG_T2.round(2),
        'pop_T1_pct'  : pop1.round(2),
        'pop_T2_pct'  : (100 - pop1).round(2),
        'dG_ref_kJ'   : np.where(np.random.rand(n) > 0.5,
                                 dG_T2 * np.random.uniform(0.8, 1.2, n),
                                 np.nan).round(2),
    })
    df_bosque.to_csv(BOSQUE_CSV, index=False)
    print(f'Bosque sintético generado: {len(df_bosque)} moléculas')

display(df_bosque.head(3))


In [ ]:
# ── Integrar la semilla al dataset ───────────────────────────────────────────
# El análisis estadístico tiene más valor cuando incluye tus propios resultados.

if len(df_boltz) >= 2:
    dG_T2_sem = df_boltz['dG_kJ'].iloc[1]
    pop1_sem  = df_boltz['pct'].iloc[0]
    ntau_sem  = len(df_boltz)
else:
    dG_T2_sem, pop1_sem, ntau_sem = 6.3, 92.5, 2   # fallback

fila_semilla = {'id': '2-piridona', 'familia': 'heterociclo_ceto_enol',
                'n_tautomeros': ntau_sem,
                'dG_T2_kJ': round(dG_T2_sem, 2),
                'pop_T1_pct': round(pop1_sem, 2),
                'pop_T2_pct': round(100 - pop1_sem, 2),
                'dG_ref_kJ': 6.3}   # Beak, 1976

df_final = pd.concat([df_bosque,
                      pd.DataFrame([fila_semilla])],
                     ignore_index=True)
df_final.to_csv('p02_dataset_final.csv', index=False)
print(f'Dataset final: {len(df_final)} moléculas (bosque + semilla)')
print(f'Semilla: ΔG_T2 = {dG_T2_sem:.2f} kJ/mol, p₁ = {pop1_sem:.1f} %')


---
## 📊 Análisis del bosque

Cada figura responde una pregunta química concreta.
**Formula tu predicción antes de ejecutar** y luego compara con el resultado.

> *¿Qué fracción de las moléculas farmacológicamente relevantes tiene un tautómero
> tan claramente dominante que podemos ignorar el resto? ¿Y en qué familias
> la incertidumbre tautómera es suficientemente grande como para invalidar
> un cálculo de propiedades?*


In [ ]:
# ── Estadística descriptiva del bosque ───────────────────────────────────────
print('=== Estadística descriptiva ===')
print(df_final[['n_tautomeros', 'dG_T2_kJ', 'pop_T1_pct']].describe().round(2))
print()
print('Umbral de seguridad p₁ > 95 %:')
for umbral in [99, 95, 80, 60]:
    n = (df_final['pop_T1_pct'] > umbral).sum()
    print(f'  p₁ > {umbral} %: {n}/{len(df_final)} moléculas')
print()
print('=== Media de p₁ por familia ===')
print(df_final.groupby('familia')['pop_T1_pct'].agg(['mean', 'min', 'max']).round(1))


---
### 📈 Figura 1 — Distribución de $p_1$ en el bosque completo

**Pregunta química:** ¿Qué fracción de las moléculas tiene un tautómero
dominante claro ($p_1 > 95$ %)? ¿En cuántas la incertidumbre es alta ($p_1 < 80$ %)?

El código de colores comunica directamente la 'seguridad' de la asignación:
- **Azul oscuro** ($p_1 > 95$ %): seguro usar ese tautómero en cálculos posteriores
- **Azul medio** (80–95 %): predomina pero la forma alternativa no es despreciable
- **Gris claro** ($p_1 < 80$ %): incertidumbre alta; se recomienda considerar múltiples formas


In [ ]:
# ── Figura 1: p_1 para el bosque completo ────────────────────────────────────
df_plot  = df_final.sort_values('pop_T1_pct').reset_index(drop=True)
colores  = ['#0F2747' if p > 95 else '#2A6F97' if p > 80 else '#AABBD0'
            for p in df_plot['pop_T1_pct']]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(df_plot)), df_plot['pop_T1_pct'],
       color=colores, edgecolor='white', linewidth=0.5)
ax.axhline(95, color='#555', linestyle='--', linewidth=0.9)
ax.axhline(80, color='#888', linestyle=':',  linewidth=0.9)

# Anotar la semilla
idx_s = df_plot.index[df_plot['id'] == '2-piridona'].tolist()
if idx_s:
    i = idx_s[0]
    ax.annotate('← semilla\n(2-piridona)',
                xy=(i, df_plot.loc[i, 'pop_T1_pct']),
                xytext=(i + 2, df_plot.loc[i, 'pop_T1_pct'] - 15),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2), fontsize=8)

n_95 = (df_final['pop_T1_pct'] > 95).sum()
n_80 = (df_final['pop_T1_pct'] <= 80).sum()
leyenda = [mpatches.Patch(color='#0F2747', label=f'p₁ > 95 %  (N = {n_95})'),
           mpatches.Patch(color='#2A6F97', label='80 % < p₁ ≤ 95 %'),
           mpatches.Patch(color='#AABBD0', label=f'p₁ ≤ 80 %  (N = {n_80})')]
ax.legend(handles=leyenda, fontsize=8, loc='lower right')
ax.set_xlabel('Molécula (ordenada por p₁ ascendente)', fontsize=11)
ax.set_ylabel('p₁ (%)', fontsize=11)
ax.set_title('Distribución de Boltzmann — Bosque P02 (40 moléculas)', fontsize=12)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('p02_poblaciones_barra.pdf', bbox_inches='tight')
plt.show()
print('✓ Guardada: p02_poblaciones_barra.pdf')


---
### 📈 Figura 2 — $\Delta G_{T_2}$ vs $p_1$, coloreada por familia

**Pregunta química:** ¿Existe una familia estructural que concentre los casos
de mayor incertidumbre tautómera? ¿Las bases nucleicas o los ceto-enoles?

La relación $\Delta G$ vs $p_1$ sigue la curva de Boltzmann teórica (línea
punteada). Los puntos que se desvían de ella tienen más de dos tautómeros
relevantes.


In [ ]:
# ── Figura 2: ΔG_T2 vs p₁ por familia ───────────────────────────────────────
paleta = {'base_nucleica'        : '#E74C3C',
          'farmaco'              : '#3498DB',
          'heterociclo'          : '#27AE60',
          'ceto_enol'            : '#F39C12',
          'heterociclo_ceto_enol': '#8E44AD'}

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Boltzmann teórica para dos formas en equilibrio
dG_rng = np.linspace(0, 35, 300)
ax.plot(dG_rng, 100 / (1 + np.exp(-dG_rng / (R * T_K))),
        'k--', linewidth=1, alpha=0.4, label='Boltzmann teórico (2 formas)')

for fam in df_final['familia'].unique():
    sub   = df_final[df_final['familia'] == fam].dropna(subset=['dG_T2_kJ'])
    color = paleta.get(fam, '#95A5A6')
    ax.scatter(sub['dG_T2_kJ'], sub['pop_T1_pct'],
               label=fam, color=color, alpha=0.85, s=55,
               edgecolors='k', linewidths=0.4)

ax.set_xlabel('ΔG_T2 (kJ/mol)', fontsize=11)
ax.set_ylabel('p₁ (%)', fontsize=11)
ax.set_title('ΔG_T2 vs p₁ por familia estructural', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(40, 102)
plt.tight_layout()
plt.savefig('p02_dG_vs_pop.pdf', bbox_inches='tight')
plt.show()
print('✓ Guardada: p02_dG_vs_pop.pdf')


---
### 📈 Figura 3 — Validación: $\Delta G$ calculado vs experimental

**Pregunta química:** ¿Dónde falla GFN2-xTB respecto a los datos experimentales?
¿El error es sistemático (bias) o aleatorio?

En un **gráfico de paridad** los puntos sobre la diagonal indican acuerdo
perfecto; los puntos por encima significan que el cálculo sobreestima $\Delta G$.
La banda ±3 kJ/mol es el criterio de aceptación para xTB en heterociclos.


In [ ]:
# ── Figura 3: paridad calculado vs experimental ──────────────────────────────
df_val = df_final.dropna(subset=['dG_ref_kJ', 'dG_T2_kJ']).copy()
print(f'Moléculas con referencia experimental: {len(df_val)}')

if len(df_val) >= 3:
    mae  = np.mean(np.abs(df_val['dG_T2_kJ'] - df_val['dG_ref_kJ']))
    r, pval = sp_stats.pearsonr(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'])

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'],
               color='#2A6F97', edgecolors='k', linewidths=0.5, s=65, zorder=5)

    lim = max(abs(df_val[['dG_ref_kJ', 'dG_T2_kJ']].values.flatten())) + 3
    ax.plot([-1, lim], [-1, lim], 'k--', linewidth=1, label='y = x')
    ax.fill_between([-1, lim], [-4, lim-3], [2, lim+3],
                    alpha=0.1, color='#2A6F97', label='±3 kJ/mol')
    ax.set_xlim(-1, lim); ax.set_ylim(-1, lim)
    ax.set_xlabel('ΔG exp. (kJ/mol)', fontsize=11)
    ax.set_ylabel('ΔG calc. GFN2-xTB (kJ/mol)', fontsize=11)
    ax.set_title(f'Paridad  |  MAE = {mae:.1f} kJ/mol  |  r = {r:.3f}', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('p02_paridad.pdf', bbox_inches='tight')
    plt.show()

    print(f'MAE = {mae:.2f} kJ/mol')
    print(f'r de Pearson = {r:.3f} (p = {pval:.4f})')
    print(f'Criterio MAE < 3 kJ/mol: {"cumplido ✓" if mae < 3 else "no cumplido ⚠"}')
    print('✓ Guardada: p02_paridad.pdf')
else:
    print('Insuficientes datos experimentales para la figura de paridad.')


---
## 🖥️ Tablero interactivo — Explorador del microespacio

El tablero siguiente es completamente interactivo:
- Pasa el cursor sobre cada punto para ver nombre, familia, ΔG y $p_1$
- Haz clic en la leyenda para mostrar/ocultar familias
- Selecciona una región para hacer zoom; doble clic para restablecer

Esta interactividad es esencial para identificar los outliers: la molécula
con mayor incertidumbre, la de mayor número de tautómeros, las que más
se desvían del experimento.


In [ ]:
# ── Tablero interactivo Plotly: dispersión + violin por familia ───────────────
if PLOTLY_OK:
    df_pl          = df_final.dropna(subset=['dG_T2_kJ']).copy()
    familias_unicas = df_pl['familia'].unique()
    paleta_px       = px.colors.qualitative.Set2

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['ΔG_T2 vs p₁ por familia', 'Distribución de p₁ (violín)'],
        horizontal_spacing=0.12
    )

    for i, fam in enumerate(familias_unicas):
        sub   = df_pl[df_pl['familia'] == fam]
        color = paleta_px[i % len(paleta_px)]
        fig.add_trace(go.Scatter(
            x=sub['dG_T2_kJ'], y=sub['pop_T1_pct'],
            mode='markers', name=fam,
            marker=dict(color=color, size=9, line=dict(color='black', width=0.5)),
            customdata=sub[['id', 'n_tautomeros', 'pop_T2_pct']].values,
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'ΔG_T2 = %{x:.1f} kJ/mol<br>'
                'p₁ = %{y:.1f} %<br>'
                'p₂ = %{customdata[2]:.1f} %<br>'
                'N tautómeros = %{customdata[1]}<extra></extra>'
            ),
            legendgroup=fam,
        ), row=1, col=1)
        fig.add_trace(go.Violin(
            y=sub['pop_T1_pct'], x=[fam]*len(sub),
            name=fam, fillcolor=color, opacity=0.5,
            box_visible=True, meanline_visible=True,
            showlegend=False, legendgroup=fam,
            line_color='black', line_width=1,
        ), row=1, col=2)

    for umbral, col_l, lbl in [(95, '#333', '95 %'), (80, '#666', '80 %')]:
        fig.add_hline(y=umbral, line_dash='dash', line_color=col_l,
                      annotation_text=lbl, row=1, col=1)

    fig.update_layout(
        title_text='<b>Tablero P02 — Microespacio de tautómeros</b>',
        title_font_size=16, height=480, template='plotly_white',
        legend=dict(title='Familia', font_size=11), violinmode='overlay'
    )
    fig.update_xaxes(title_text='ΔG_T2 (kJ/mol)', row=1, col=1)
    fig.update_yaxes(title_text='p₁ (%)', row=1, col=1)
    fig.update_xaxes(tickangle=20, row=1, col=2)
    fig.show()
    fig.write_html('p02_tablero_interactivo.html')
    print('✓ Guardado: p02_tablero_interactivo.html')
else:
    print('Plotly no disponible. Instala con: pip install plotly')


---
## 🔄 Revisión de hipótesis iniciales

Compara tus respuestas previas con los resultados obtenidos.
Esta reflexión es el corazón del método científico: contrastar predicciones
con evidencia.


In [ ]:
# ── Comparar predicciones iniciales con resultados ───────────────────────────
print('=' * 60)
print('  TUS RESPUESTAS INICIALES')
print('=' * 60)
print('\nPregunta 1 (guanina):',    respuesta_1)
print('\nPregunta 2 (ceto vs enol):', respuesta_2)
print('\nPregunta 3 (limitaciones):', respuesta_3)

print()
print('=' * 60)
print('  RESULTADOS OBTENIDOS')
print('=' * 60)
if len(df_boltz) > 0:
    dom = df_boltz.iloc[0]
    print(f'  Tautómero dominante : {dom["etiqueta"]} ({dom["pct"]:.1f} %)')
    if len(df_boltz) > 1:
        print(f'  ΔG calculado        : {df_boltz["dG_kJ"].iloc[1]:.2f} kJ/mol'
              f' (ref. Beak 1976: 6.3 kJ/mol)')
print(f'  Familia con mayor incertidumbre: '
      f'{df_final.groupby("familia")["pop_T1_pct"].mean().idxmin()}')


In [ ]:
# ── Reflexión final ───────────────────────────────────────────────────────────
reflexion_final = (
    '¿Tu predicción sobre la forma dominante fue correcta?:\n'
    '\n'
    '¿Qué fue lo más sorprendente del análisis del bosque?:\n'
    '\n'
    '¿Qué limitación del protocolo en vacío observaste claramente?:\n'
    '\n'
    '¿Qué pregunta nueva te generó esta práctica?:\n'
)
print(reflexion_final)


---
## 📦 Entregables

| Tipo | Archivo | Descripción |
|:--|:--|:--|
| **D1** | `2-piridona_tautomeros_resultados.csv` | G, ΔG y poblaciones de la semilla |
| **D2** | `p02_dataset_final.csv` | Bosque completo + semilla integrada |
| **D3** | `2-piridona_T01_xtb.out`, `T02_xtb.out` | Logs xTB de los dos tautómeros principales |
| **F1** | `p02_boltzmann_semilla.pdf` | ΔG + distribución de poblaciones (semilla) |
| **F2** | `p02_pop_vs_T.pdf` | Población vs temperatura (200–800 K) |
| **F3** | `p02_poblaciones_barra.pdf` | Distribución de $p_1$ en el bosque |
| **F4** | `p02_dG_vs_pop.pdf` | Dispersión $\Delta G_{T_2}$ vs $p_1$ por familia |
| **F5** | `p02_paridad.pdf` | Gráfico de paridad calc. vs exp. |
| **F6** | `p02_tablero_interactivo.html` | Tablero Plotly interactivo |
| **T1** | Reporte (≤ 2 páginas) | Ver especificaciones abajo |

**Especificaciones del reporte T1:**
- Tabla de validación geométrica de la semilla vs referencia (Beak, 1976)
- F1 y F3 comentadas con interpretación química (no solo descripción)
- Respuestas a las preguntas de discusión (≤ 150 palabras c/u)
- Reflexión sobre las limitaciones del protocolo en vacío


In [ ]:
# ── Checklist automático de entregables ──────────────────────────────────────
entregables = [
    ('D1', f'{NOMBRE}_tautomeros_resultados.csv'),
    ('D2', 'p02_dataset_final.csv'),
    ('D3', f'{DIRECTORIO}/{NOMBRE}_T01_xtb.out'),
    ('D3', f'{DIRECTORIO}/{NOMBRE}_T02_xtb.out'),
    ('F1', 'p02_boltzmann_semilla.pdf'),
    ('F2', 'p02_pop_vs_T.pdf'),
    ('F3', 'p02_poblaciones_barra.pdf'),
    ('F4', 'p02_dG_vs_pop.pdf'),
    ('F5', 'p02_paridad.pdf'),
    ('F6', 'p02_tablero_interactivo.html'),
]

print('=== Checklist de entregables ===')
todos_ok = True
for tipo, archivo in entregables:
    p      = pathlib.Path(archivo)
    existe = p.exists() and p.stat().st_size > 0
    estado = '✓ OK' if existe else '✗ FALTA'
    if not existe:
        todos_ok = False
    print(f'  [{estado}] {tipo}: {archivo}')

print()
print('✓ Todos los entregables listos.' if todos_ok else
      '⚠ Faltan entregables — revisa las celdas anteriores.')


---
## ❓ Preguntas de discusión

Responde en el reporte T1 (≤ 150 palabras por pregunta).
Usa la celda siguiente para esbozar ideas antes de redactar.


In [ ]:
preguntas = {
    1: ('(Comprensión) ¿Cuántos tautómeros encontró TautomerEnumerator para la '
        '2-piridona? ¿Son todos químicamente razonables? Si alguno no lo es, '
        'explica por qué el algoritmo lo genera igualmente. '
        '¿Qué tipo de tautomería podría NOT detectar?'),

    2: ('(Comprensión) Localiza en el log xTB las líneas TOTAL ENERGY y '
        'TOTAL FREE ENERGY. ¿Cuánto difieren en kJ/mol? '
        '¿Qué contribuciones energéticas suma xTB para pasar de E a G?'),

    3: ('(Análisis) Usa la ecuación de Boltzmann para calcular manualmente '
        'qué ΔG es necesario para que el tautómero minoritario tenga exactamente '
        '1 % de población. ¿A qué temperatura sería observable con RMN estándar?'),

    4: ('(Análisis) Del bosque, identifica la molécula con la mayor incertidumbre '
        'tautómera (menor ΔG_T2). ¿Qué consecuencia tendría para un cálculo de '
        'docking ejecutado sobre el tautómero incorrecto?'),

    5: ('(Metodológico) El protocolo usa energías en vacío. '
        '¿En qué dirección se desplazaría el equilibrio 2-piridona ⇌ '
        '2-hidroxipiridina al pasar de vacío a agua? ¿Y a ciclohexano? '
        'Justifica con polaridad y capacidad de formar puentes de hidrógeno.'),

    6: ('(SMARTS) Escribe el SMARTS para buscar grupos amidina C(=NH)N '
        'en un dataset de SMILES. ¿Por qué es útil este patrón para '
        'identificar moléculas propensas a tautomería?'),
}

for num, texto in preguntas.items():
    print(f'P{num}. {texto}')
    print()


---
## 🔗 Conexión con el resto del manual

```{admonition} Guarda estos archivos — los usarás en prácticas futuras
:class: tip

- `2-piridona_T01_opt.xyz` → input directo de la **Práctica 4**
  (el tautómero dominante es el punto de partida correcto para DFT)
- `p02_dataset_final.csv` → referencia para la **Práctica 30**
  (efecto del solvente PCM/SMD sobre equilibrios tautómeros)
```

| Habilidad adquirida | Se usa en… |
|:--|:--|
| `TautomerEnumerator` + SMARTS | P04+: input correcto para todos los cálculos DFT |
| Distribución de Boltzmann | P08 (conformaciones), P15 (rotámeros) |
| $p_i(T)$ — Boltzmann vs temperatura | P03, P17: equilibrios a otras condiciones |
| Lectura de logs xTB | Habilidad central de todo el Bloque 1 |
| Gráficos de paridad | Estándar de validación en todo el manual |
| Efecto del solvente (ausente aquí) | P30: PCM/SMD cuantificará el desplazamiento |
| `p02_dataset_final.csv` | Input para P47 (exploración del espacio químico) |
| SELFIES | P48–P50 (generación molecular con REINVENT) |

---

## 📌 Nota al manual — recursos a añadir

> ✏️ **Para el docente:** los siguientes recursos enriquecerían esta práctica
> y están pendientes de creación:

> 1. **Diagrama de cuatro cuadrantes SMILES/SMARTS/SELFIES/estructura 2D**
>    con la 2-piridona como ejemplo. Insertar en la sección de cuadro conceptual.

> 2. **Figura de las dos estructuras Lewis** (2-piridona y 2-hidroxipiridina)
>    con el protón migrante resaltado y las distancias de enlace C=O/C–OH anotadas
>    (calc vs exp). Insertar al inicio del Paso 1.

> 3. **Mapa del bosque** (panel 4×2): un representante por familia con estructura 2D
>    y su $p_1$ anotado. Insertar antes del análisis estadístico.

> 4. **Perfil de energía libre** (coordenada de reacción): los dos tautómeros como
>    mínimos con la barrera entre ellos. Conectaría la práctica con cinética.

> 5. **Tabla de tipos de tautomería** con patrones SMARTS y grupos funcionales
>    representativos: recurso de consulta rápida valioso para el Anexo B.

> 6. **Párrafo sobre SELFIES** en el marco teórico del `.tex` con referencia a
>    Krenn et al. (2020), `doi:10.1088/2632-2153/aba947`.
